# Playground S6E5 — Predicting F1 Pit Stops
Leaderboard-odaklı, leakage-free pipeline.

**Strateji**
1. `race_id = Year_Race` üzerinden `StratifiedGroupKFold` ile gerçek CV.
2. Stint-bazlı, sıralı (no future leakage) feature engineering.
3. LightGBM + XGBoost + CatBoost OOF; rank-average ensemble.
4. Optuna iskeleti (opsiyonel), seed manager, tek `submission.csv` çıkışı.

Metric: yarışma **binary log loss / ROC AUC** (her ikisini de takip ediyoruz).

## 1. Setup

In [ ]:
# Kaggle'da lightgbm/xgboost/catboost zaten kurulu. Lokalde eksikse aç:
# !pip install -q lightgbm xgboost catboost scikit-learn optuna

import os, gc, math, random, warnings, json
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import log_loss, roc_auc_score

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier, Pool

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 120)

SEED = 42
N_SPLITS = 5

def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed); os.environ['PYTHONHASHSEED'] = str(seed)
seed_everything()

# Path resolver: Kaggle ya da local
CANDIDATE_DIRS = [
    Path('/kaggle/input/playground-series-s6e5'),
    Path('/kaggle/input/competitions/playground-series-s6e5'),
    Path('data'),
    Path('/Users/berk.terekli/Documents/GitHub/kaggle/predicting-f1-pit-stops/data'),
]
DATA_DIR = next((p for p in CANDIDATE_DIRS if (p / 'train.csv').exists()), None)
assert DATA_DIR is not None, 'train.csv bulunamadı'
print('DATA_DIR:', DATA_DIR)

ID_COL = 'id'
TARGET = 'PitNextLap'


## 2. Load & Memory Optimize

In [ ]:
def reduce_mem(df):
    for c in df.select_dtypes(include=['float64']).columns:
        df[c] = df[c].astype('float32')
    for c in df.select_dtypes(include=['int64']).columns:
        cmin, cmax = df[c].min(), df[c].max()
        if cmin >= np.iinfo(np.int32).min and cmax <= np.iinfo(np.int32).max:
            df[c] = df[c].astype('int32')
    return df

train = reduce_mem(pd.read_csv(DATA_DIR / 'train.csv'))
test  = reduce_mem(pd.read_csv(DATA_DIR / 'test.csv'))
sub   = pd.read_csv(DATA_DIR / 'sample_submission.csv')
print('train:', train.shape, 'test:', test.shape)
print('target rate:', train[TARGET].mean().round(5))
train.head(3)

## 3. Leakage Check
`PitNextLap = 1` ise sıradaki lap pit. `PitStop` mevcut lap'in pit olup olmadığı; test'te de verili → güvenli kullan.
Sıralama: `(race_id, Driver, LapNumber)`. Tüm lag/rolling işlemleri `shift(1)` ile alınır → next-lap leakage yok.

In [ ]:
# Train'de pit-this-lap ile pit-next-lap korelasyonu (sanity)
print(pd.crosstab(train['PitStop'], train[TARGET], normalize='index'))

## 4. Feature Engineering (leakage-free)

In [ ]:
LAPTIME = 'LapTime (s)'

def add_group_keys(df):
    df = df.copy()
    df['race_id']   = df['Year'].astype(str) + '_' + df['Race'].astype(str)
    df['driver_race_id'] = df['race_id'] + '_' + df['Driver'].astype(str)
    df['stint_id']  = df['driver_race_id'] + '_' + df['Stint'].astype(str)
    return df

train = add_group_keys(train)
test  = add_group_keys(test)

# Tek dataframe üzerinde feature üret (train+test). Target sadece train'de.
full = pd.concat([train.assign(_is_train=1), test.assign(_is_train=0, **{TARGET: np.nan})], ignore_index=True)
full = full.sort_values(['race_id', 'Driver', 'LapNumber']).reset_index(drop=True)

g_dr = full.groupby('driver_race_id', sort=False)
g_st = full.groupby('stint_id', sort=False)
g_rc = full.groupby('race_id', sort=False)

# --- Race-level meta (no leakage; aynı race tüm satırlarda var) ---
full['race_total_laps']  = g_rc['LapNumber'].transform('max')
full['laps_remaining']   = full['race_total_laps'] - full['LapNumber']
full['race_drivers']     = g_rc['Driver'].transform('nunique')

# --- Stint dynamics ---
full['lap_in_stint']      = g_st.cumcount() + 1            # 1-indexed
full['stint_max_tyre']    = g_st['TyreLife'].transform('max')
full['tyre_vs_max']       = full['TyreLife'] / (full['stint_max_tyre'] + 1e-6)

# --- Driver-race pit history so far (only past laps -> cumsum then shift) ---
full['pits_so_far'] = g_dr['PitStop'].cumsum().astype('float32') - full['PitStop']  # exclude current
full['stint_index'] = full['Stint']

# --- Lag / rolling features (shifted to avoid leakage) ---
def lag(col, k, grp=g_dr):
    return grp[col].shift(k)

for k in [1, 2, 3]:
    full[f'laptime_lag{k}']     = lag(LAPTIME, k)
    full[f'laptime_delta_lag{k}']  = lag('LapTime_Delta', k)
    full[f'pos_change_lag{k}']     = lag('Position_Change', k)

# Rolling on past 3 laps (shift(1) ensures no current value leaked)
def roll_mean(col, w):
    return g_dr[col].apply(lambda s: s.shift(1).rolling(w, min_periods=1).mean()).reset_index(level=0, drop=True)
def roll_std(col, w):
    return g_dr[col].apply(lambda s: s.shift(1).rolling(w, min_periods=2).std()).reset_index(level=0, drop=True)

for w in [3, 5]:
    full[f'laptime_rmean_{w}']     = roll_mean(LAPTIME, w)
    full[f'laptime_rstd_{w}']      = roll_std(LAPTIME, w)
    full[f'delta_rmean_{w}']       = roll_mean('LapTime_Delta', w)
    full[f'degr_rmean_{w}']        = roll_mean('Cumulative_Degradation', w)

# Pace trend: slope-like simple feature
full['laptime_trend_3'] = full['laptime_lag1'] - full['laptime_lag3']
full['delta_trend_3']   = full['laptime_delta_lag1'] - full['laptime_delta_lag3']

# --- Interaction features ---
full['tyre_x_progress']    = full['TyreLife'] * full['RaceProgress']
full['degr_per_tyre']      = full['Cumulative_Degradation'] / (full['TyreLife'] + 1e-3)
full['lapsrem_x_tyre']     = full['laps_remaining'] * full['TyreLife']

# --- Compound stint length stats (TRAIN-ONLY to avoid leakage) ---
tr_mask = full['_is_train'] == 1
comp_stint = (full[tr_mask].groupby(['Compound', 'stint_id'])['TyreLife'].max()
                .groupby('Compound').agg(['mean', 'std']).rename(columns={'mean':'comp_avg_stint','std':'comp_std_stint'}))
full = full.merge(comp_stint, on='Compound', how='left')
full['tyre_vs_comp_avg'] = full['TyreLife'] / (full['comp_avg_stint'] + 1e-6)

# --- Categorical encoding ---
CAT_COLS = ['Driver', 'Compound', 'Race']
for c in CAT_COLS:
    full[c] = full[c].astype('category')

# Split back
train_fe = full[full['_is_train'] == 1].reset_index(drop=True).drop(columns=['_is_train'])
test_fe  = full[full['_is_train'] == 0].reset_index(drop=True).drop(columns=['_is_train', TARGET])
del full; gc.collect()

train_fe = train_fe.sort_values(ID_COL).reset_index(drop=True)
test_fe  = test_fe.sort_values(ID_COL).reset_index(drop=True)
print('FE shapes:', train_fe.shape, test_fe.shape)

## 5. OOF Target Encoding (Driver pit propensity)
Driver bazlı pit oranı güçlü bir sinyal. Leakage'i önlemek için **fold-out** ortalama; test'e ise full-train ortalamasını veriyoruz.

In [ ]:
GROUP_COL = 'race_id'
groups = train_fe[GROUP_COL].values
y = train_fe[TARGET].astype('int8').values

sgkf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
folds = list(sgkf.split(train_fe, y, groups))

def oof_target_encode(col, smoothing=20.0):
    oof = np.full(len(train_fe), np.nan, dtype='float32')
    global_mean = float(y.mean())
    # Categorical kolonu string'e çevirip map'le (yoksa fillna patlar)
    tr_col_full = train_fe[col].astype(str)
    te_col_full = test_fe[col].astype(str)
    tmp = train_fe[[col, TARGET]].copy()
    tmp[col] = tr_col_full
    for tr_idx, va_idx in folds:
        agg = tmp.iloc[tr_idx].groupby(col)[TARGET].agg(['mean', 'count'])
        smooth = (agg['mean'] * agg['count'] + global_mean * smoothing) / (agg['count'] + smoothing)
        mapped = tr_col_full.iloc[va_idx].map(smooth).astype('float32')
        oof[va_idx] = mapped.fillna(global_mean).values
    full_agg = tmp.groupby(col)[TARGET].agg(['mean', 'count'])
    full_smooth = (full_agg['mean'] * full_agg['count'] + global_mean * smoothing) / (full_agg['count'] + smoothing)
    test_enc = te_col_full.map(full_smooth).astype('float32').fillna(global_mean).values
    return oof, test_enc

for col in ['Driver', 'Compound', 'Race']:
    oof_enc, te_enc = oof_target_encode(col)
    train_fe[f'te_{col}'] = oof_enc
    test_fe[f'te_{col}']  = te_enc
print('Target encoding done.')


## 6. Feature Set

In [ ]:
DROP = {ID_COL, TARGET, 'race_id', 'driver_race_id', 'stint_id'}
FEATURES = [c for c in train_fe.columns if c not in DROP]
CAT_FEATURES = [c for c in CAT_COLS if c in FEATURES]
print(f'{len(FEATURES)} features, {len(CAT_FEATURES)} categorical')
print(FEATURES)

## 7. CV Training — LightGBM

In [ ]:
lgb_params = dict(
    objective='binary', metric='binary_logloss',
    learning_rate=0.03, num_leaves=255, min_data_in_leaf=80,
    feature_fraction=0.85, bagging_fraction=0.85, bagging_freq=1,
    lambda_l1=0.1, lambda_l2=0.1, max_depth=-1,
    verbose=-1, seed=SEED, n_jobs=-1,
)

oof_lgb = np.zeros(len(train_fe), dtype='float32')
pred_lgb = np.zeros(len(test_fe), dtype='float32')

X = train_fe[FEATURES]
Xt = test_fe[FEATURES]

for f, (tr_idx, va_idx) in enumerate(folds):
    dtr = lgb.Dataset(X.iloc[tr_idx], y[tr_idx], categorical_feature=CAT_FEATURES)
    dva = lgb.Dataset(X.iloc[va_idx], y[va_idx], categorical_feature=CAT_FEATURES)
    model = lgb.train(lgb_params, dtr, num_boost_round=6000,
                      valid_sets=[dva],
                      callbacks=[lgb.early_stopping(150), lgb.log_evaluation(0)])
    oof_lgb[va_idx] = model.predict(X.iloc[va_idx], num_iteration=model.best_iteration)
    pred_lgb += model.predict(Xt, num_iteration=model.best_iteration) / N_SPLITS
    print(f'[LGB f{f}] best_iter={model.best_iteration} '
          f'logloss={log_loss(y[va_idx], oof_lgb[va_idx]):.5f} '
          f'auc={roc_auc_score(y[va_idx], oof_lgb[va_idx]):.5f}')

print(f'OOF LGB | logloss={log_loss(y, oof_lgb):.5f} auc={roc_auc_score(y, oof_lgb):.5f}')

## 8. CV Training — XGBoost

In [ ]:
xgb_params = dict(
    objective='binary:logistic', eval_metric='logloss', tree_method='hist',
    learning_rate=0.04, max_depth=8, min_child_weight=10,
    subsample=0.85, colsample_bytree=0.85, reg_lambda=1.0, reg_alpha=0.1,
    seed=SEED, n_jobs=-1,
    # device='cuda',  # Kaggle GPU notebook'ta açın
    enable_categorical=True,
)

oof_xgb = np.zeros(len(train_fe), dtype='float32')
pred_xgb = np.zeros(len(test_fe), dtype='float32')

for f, (tr_idx, va_idx) in enumerate(folds):
    dtr = xgb.DMatrix(X.iloc[tr_idx], label=y[tr_idx], enable_categorical=True)
    dva = xgb.DMatrix(X.iloc[va_idx], label=y[va_idx], enable_categorical=True)
    dte = xgb.DMatrix(Xt, enable_categorical=True)
    model = xgb.train(xgb_params, dtr, num_boost_round=6000,
                      evals=[(dva, 'va')], early_stopping_rounds=150, verbose_eval=0)
    oof_xgb[va_idx] = model.predict(dva, iteration_range=(0, model.best_iteration+1))
    pred_xgb += model.predict(dte, iteration_range=(0, model.best_iteration+1)) / N_SPLITS
    print(f'[XGB f{f}] best_iter={model.best_iteration} '
          f'logloss={log_loss(y[va_idx], oof_xgb[va_idx]):.5f} '
          f'auc={roc_auc_score(y[va_idx], oof_xgb[va_idx]):.5f}')

print(f'OOF XGB | logloss={log_loss(y, oof_xgb):.5f} auc={roc_auc_score(y, oof_xgb):.5f}')

## 9. CV Training — CatBoost

In [ ]:
cb_params = dict(
    loss_function='Logloss', eval_metric='Logloss',
    iterations=6000, learning_rate=0.05, depth=8,
    l2_leaf_reg=5.0, random_seed=SEED, verbose=0,
    # task_type='GPU',  # Kaggle GPU
)

oof_cb = np.zeros(len(train_fe), dtype='float32')
pred_cb = np.zeros(len(test_fe), dtype='float32')

# CatBoost native cat: ham string istiyor, NaN'a izin var
X_cb  = train_fe[FEATURES].copy()
Xt_cb = test_fe[FEATURES].copy()
for c in CAT_FEATURES:
    X_cb[c]  = X_cb[c].astype(str)
    Xt_cb[c] = Xt_cb[c].astype(str)

for f, (tr_idx, va_idx) in enumerate(folds):
    tr_pool = Pool(X_cb.iloc[tr_idx], y[tr_idx], cat_features=CAT_FEATURES)
    va_pool = Pool(X_cb.iloc[va_idx], y[va_idx], cat_features=CAT_FEATURES)
    te_pool = Pool(Xt_cb,             cat_features=CAT_FEATURES)
    model = CatBoostClassifier(**cb_params)
    model.fit(tr_pool, eval_set=va_pool, early_stopping_rounds=200, use_best_model=True, verbose=0)
    oof_cb[va_idx] = model.predict_proba(va_pool)[:, 1]
    pred_cb += model.predict_proba(te_pool)[:, 1] / N_SPLITS
    print(f'[CB  f{f}] best_iter={model.get_best_iteration()} '
          f'logloss={log_loss(y[va_idx], oof_cb[va_idx]):.5f} '
          f'auc={roc_auc_score(y[va_idx], oof_cb[va_idx]):.5f}')

print(f'OOF CB  | logloss={log_loss(y, oof_cb):.5f} auc={roc_auc_score(y, oof_cb):.5f}')

## 10. Ensemble: Logit Mean + Rank Mean
Üç model logit ortalaması (logloss için doğal), rank ortalaması (AUC için sağlam). Final = ikisinin tekrar logit ortalaması.

In [ ]:
EPS = 1e-7
def logit(p):
    p = np.clip(p, EPS, 1 - EPS); return np.log(p / (1 - p))
def sigmoid(z): return 1.0 / (1.0 + np.exp(-z))
def rank01(x):
    r = pd.Series(x).rank(method='average').values
    return (r - 1) / (len(r) - 1)

# Weight search on OOF (logloss)
from itertools import product
best = (1e9, None)
for w in product(np.linspace(0, 1, 11), repeat=3):
    if abs(sum(w) - 1) > 1e-6: continue
    blend = sigmoid(w[0]*logit(oof_lgb) + w[1]*logit(oof_xgb) + w[2]*logit(oof_cb))
    ll = log_loss(y, blend)
    if ll < best[0]: best = (ll, w)
print('Best weights (lgb,xgb,cb):', best[1], 'OOF logloss:', round(best[0], 6))
wl, wx, wc = best[1]

oof_logit  = sigmoid(wl*logit(oof_lgb) + wx*logit(oof_xgb) + wc*logit(oof_cb))
pred_logit = sigmoid(wl*logit(pred_lgb) + wx*logit(pred_xgb) + wc*logit(pred_cb))

oof_rank  = wl*rank01(oof_lgb)  + wx*rank01(oof_xgb)  + wc*rank01(oof_cb)
pred_rank = wl*rank01(pred_lgb) + wx*rank01(pred_xgb) + wc*rank01(pred_cb)

# AUC korumak için 50/50 prob & rank-as-prob (rank'i [eps,1-eps]'e map)
def rank_to_prob_like(r):
    return np.clip(r, EPS, 1 - EPS)

oof_final  = 0.7 * oof_logit  + 0.3 * rank_to_prob_like(oof_rank)
pred_final = 0.7 * pred_logit + 0.3 * rank_to_prob_like(pred_rank)

print(f'OOF FINAL | logloss={log_loss(y, oof_final):.5f} auc={roc_auc_score(y, oof_final):.5f}')

## 11. Submission

In [ ]:
out = pd.DataFrame({ID_COL: test_fe[ID_COL].values, TARGET: np.clip(pred_final, EPS, 1 - EPS)})
out = out.sort_values(ID_COL).reset_index(drop=True)
out.to_csv('submission.csv', index=False)

# Diagnostic artifacts
pd.DataFrame({ID_COL: test_fe[ID_COL].values,
              'lgb': pred_lgb, 'xgb': pred_xgb, 'cb': pred_cb,
              'logit_blend': pred_logit, 'rank_blend': pred_rank,
              'final': pred_final}).to_csv('test_predictions_breakdown.csv', index=False)
pd.DataFrame({ID_COL: train_fe[ID_COL].values,
              'y': y, 'oof_lgb': oof_lgb, 'oof_xgb': oof_xgb, 'oof_cb': oof_cb,
              'oof_final': oof_final}).to_csv('oof_predictions.csv', index=False)
print('Wrote submission.csv', out.shape)
out.head()

## 12. (Opsiyonel) Optuna ile LightGBM Tuning
Uzun sürer; sadece final boost için bir kez çalıştırın.

In [ ]:
RUN_OPTUNA = False
if RUN_OPTUNA:
    import optuna
    def objective(trial):
        params = dict(
            objective='binary', metric='binary_logloss', verbose=-1, seed=SEED,
            learning_rate=trial.suggest_float('lr', 0.01, 0.08, log=True),
            num_leaves=trial.suggest_int('leaves', 64, 512),
            min_data_in_leaf=trial.suggest_int('mdl', 20, 200),
            feature_fraction=trial.suggest_float('ff', 0.6, 1.0),
            bagging_fraction=trial.suggest_float('bf', 0.6, 1.0),
            bagging_freq=1,
            lambda_l1=trial.suggest_float('l1', 1e-3, 5.0, log=True),
            lambda_l2=trial.suggest_float('l2', 1e-3, 5.0, log=True),
        )
        scores = []
        for tr_idx, va_idx in folds[:3]:
            dtr = lgb.Dataset(X.iloc[tr_idx], y[tr_idx], categorical_feature=CAT_FEATURES)
            dva = lgb.Dataset(X.iloc[va_idx], y[va_idx], categorical_feature=CAT_FEATURES)
            m = lgb.train(params, dtr, 4000, valid_sets=[dva],
                          callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)])
            p = m.predict(X.iloc[va_idx], num_iteration=m.best_iteration)
            scores.append(log_loss(y[va_idx], p))
        return float(np.mean(scores))
    study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(objective, n_trials=40, show_progress_bar=True)
    print('best:', study.best_value, study.best_params)

## 13. Public Anchor Blending — Hedef > 0.95489

**Kaggle dataset bağımlılıkları (sağ panelden ekle):**
- `playground-series-s6e5` (yarışma datası)
- `raunakdey07/f1-pit-stops-0-95454` → 0.95454 anchor
- `nawfeelrahman1124444/pss6ep5448` → 0.95448 anchor
- `flexonafft/f1-submissions` → blend_dataset (0.95449 + 0.95446 rank-diverse + diğer support)

Strateji:
- En iyi tek anchor (`s54`) üstüne kendi modelimizden %5/%10/%15/%20 logit-rank katkı
- `s54 + s48 + s49` rank-average ile public diversity submission
- `s46` rank-diverse source mikro-injection
- Final canonical: `submission.csv` = anchor + model %10

**Kaggle'da hangi dosyaları submit etmeli (önerilen sıra):**
1. `submission_full_blend.csv` → en agresif kombinasyon
2. `submission_anchor_plus_model_10.csv` → en savunmacı
3. `submission_pubavg_plus_model_10.csv` → public diversity + model


In [ ]:
from pathlib import Path

# Kaggle dataset paths (kullanıcının eklediği)
BLEND_CANDIDATE_DIRS = [
    Path('/kaggle/input/datasets/flexonafft/f1-submissions/blend_dataset'),
    Path('/kaggle/input/f1-submissions/blend_dataset'),
    Path('blend_dataset'),
    Path('/Users/berk.terekli/Documents/GitHub/kaggle/predicting-f1-pit-stops/blend_dataset'),
]
BLEND_DIR = next((p for p in BLEND_CANDIDATE_DIRS if (p / 'public').exists()), None)
print('BLEND_DIR:', BLEND_DIR)

# Ekstra anchor CSV path adayları
S54_CANDIDATES = [
    Path('/kaggle/input/datasets/raunakdey07/f1-pit-stops-0-95454/submission.csv'),
    Path('/kaggle/input/f1-pit-stops-0-95454/submission.csv'),
]
S48_CANDIDATES = [
    Path('/kaggle/input/datasets/nawfeelrahman1124444/pss6ep5448/submission - 2026-05-19T092611.187.csv'),
    Path('/kaggle/input/pss6ep5448/submission - 2026-05-19T092611.187.csv'),
]

def load_pred_csv(path, ref_ids):
    df = pd.read_csv(path)
    tgt = TARGET if TARGET in df.columns else [c for c in df.columns if c != ID_COL][0]
    df = df[[ID_COL, tgt]].rename(columns={tgt: 'p'})
    df = df.set_index(ID_COL).reindex(ref_ids).reset_index()
    return df['p'].astype('float64').values

ref_ids = test_fe[ID_COL].values

anchors = {}
if BLEND_DIR is not None:
    # blend_dataset içinden anchor + rank source
    p49 = BLEND_DIR / 'public/super/0.95449.csv'
    if p49.exists():
        anchors['s49'] = load_pred_csv(p49, ref_ids)
    p46 = BLEND_DIR / 'public/rank_diverse/0.95446.csv'
    if p46.exists():
        anchors['s46'] = load_pred_csv(p46, ref_ids)

# 0.95454 anchor
for cand in S54_CANDIDATES:
    if cand.exists():
        anchors['s54'] = load_pred_csv(cand, ref_ids)
        break

# 0.95448 anchor (yeni)
for cand in S48_CANDIDATES:
    if cand.exists():
        anchors['s48'] = load_pred_csv(cand, ref_ids)
        break

print('Loaded anchors:')
for k, v in anchors.items():
    print(f'  {k}: mean={v.mean():.5f} std={v.std():.5f} n={len(v)}')


In [ ]:
# ------------- Blending helpers -------------
def normalized_rank(x):
    x = np.asarray(x, dtype='float64')
    order = np.argsort(x, kind='mergesort')
    r = np.empty_like(order, dtype='float64')
    r[order] = np.linspace(1e-6, 1 - 1e-6, len(x))
    return r

def logit_rank_blend(anchor, support, w_support):
    """Anchor'ın probability dağılımını korur, sıralamayı support ile karıştırır."""
    a_r = normalized_rank(anchor)
    s_r = normalized_rank(support)
    a_l = np.log(a_r / (1 - a_r))
    s_l = np.log(s_r / (1 - s_r))
    blended_rank = 1.0 / (1.0 + np.exp(-((1 - w_support) * a_l + w_support * s_l)))
    order = np.argsort(blended_rank, kind='mergesort')
    out = np.empty_like(anchor, dtype='float64')
    out[order] = np.sort(np.asarray(anchor, dtype='float64'))
    return np.clip(out, 1e-7, 1 - 1e-7)

def rank_avg(arrs):
    return np.mean([normalized_rank(a) for a in arrs], axis=0)

def write_sub(arr, name):
    df = pd.DataFrame({ID_COL: ref_ids, TARGET: np.clip(arr, 1e-7, 1 - 1e-7)})
    df.to_csv(name, index=False)
    print(f'wrote {name:50s}  mean={arr.mean():.5f}  std={arr.std():.5f}')

model_pred = pred_final  # cell 11'den gelen kendi ensemble tahminimiz

# 1) Saf model
write_sub(model_pred, 'submission_model_only.csv')

# 2) En iyi public anchor seçimi: s54 > s48 > s49
anchor = None
for tag in ['s54', 's48', 's49']:
    if tag in anchors:
        anchor = anchors[tag]; anchor_tag = tag; break

if anchor is None:
    print('UYARI: hiçbir public anchor yüklenmedi — sadece model_only submission var.')
    final = model_pred
else:
    print(f'Primary anchor: {anchor_tag}')

    # 3) Public anchor'lar arası rank-avg (varsa) — tek başına güçlü submission
    pub_anchors = [v for k, v in anchors.items() if k in ('s54', 's48', 's49')]
    if len(pub_anchors) >= 2:
        # En güçlü anchor'ın dağılımına diğerlerinin rank ortalamasını map et
        pub_rank = rank_avg(pub_anchors)
        order = np.argsort(pub_rank, kind='mergesort')
        pub_mix = np.empty_like(anchor)
        pub_mix[order] = np.sort(anchor)
        pub_mix = np.clip(pub_mix, 1e-7, 1 - 1e-7)
        write_sub(pub_mix, 'submission_public_rankavg.csv')
    else:
        pub_mix = anchor

    # 4) Anchor + model logit-rank blend (farklı ağırlıklar)
    for w in [0.05, 0.10, 0.15, 0.20]:
        blended = logit_rank_blend(anchor, model_pred, w_support=w)
        write_sub(blended, f'submission_anchor_plus_model_{int(w*100):02d}.csv')

    # 5) public_rankavg + model
    if len(pub_anchors) >= 2:
        for w in [0.05, 0.10, 0.15]:
            blended = logit_rank_blend(pub_mix, model_pred, w_support=w)
            write_sub(blended, f'submission_pubavg_plus_model_{int(w*100):02d}.csv')

    # 6) Anchor + model + rank_diverse micro injection
    if 's46' in anchors:
        step1 = logit_rank_blend(anchor, model_pred, w_support=0.10)
        step2 = logit_rank_blend(step1, anchors['s46'], w_support=0.02)
        write_sub(step2, 'submission_full_blend.csv')

        # Ultra-narrow gate: sadece ambiguous middle 0.2-0.8 bölgesinde s46 etkisi
        gated = step1.copy()
        mask = (anchor >= 0.20) & (anchor <= 0.80)
        gate_full = logit_rank_blend(step1, anchors['s46'], w_support=0.05)
        gated[mask] = gate_full[mask]
        write_sub(gated, 'submission_full_blend_gated.csv')

    # 7) Canonical final = en savunmacı seçim: en iyi anchor + model %10
    final = logit_rank_blend(anchor, model_pred, w_support=0.10)

write_sub(final, 'submission.csv')

# Diagnostic: anchor'lar arası korelasyon
if len(anchors) > 0:
    import itertools
    corr_rows = []
    keys = list(anchors.keys()) + ['model']
    arrs = list(anchors.values()) + [model_pred]
    for i, j in itertools.combinations(range(len(keys)), 2):
        c = np.corrcoef(arrs[i], arrs[j])[0, 1]
        corr_rows.append((keys[i], keys[j], round(float(c), 5)))
    print('\nPairwise correlations:')
    for r in corr_rows:
        print(f'  {r[0]:6s} vs {r[1]:6s}: {r[2]}')
